# MosaicDiff Web Service on Google Colab

Notebook ini disiapkan untuk menjalankan **MosaicDiff** sebagai service web (Gradio) menggunakan GPU A100 di Google Colab Pro, dan di-tunnel ke AWS VM Anda.

> **PENTING:** Pastikan Anda memilih Hardware accelerator **GPU A100** dan **High-RAM** (Runtime > Change runtime type).


### 1. Cek GPU dan Alokasi VRAM
Pastikan GPU yang terdeteksi adalah A100 atau L4 dengan VRAM yang mencukupi.

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM: {total_mem:.2f} GB')

### 2. Setup ComfyUI & MosaicDiff Workspace
ComfyUI diperlukan sebagai backend inferensi untuk model MiniMax H3.

In [ ]:
# Clone ComfyUI backend
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt

# Pindah ke folder MosaicDiff (sesuaikan jika folder diunggah ke path lain)
%cd /content/MosaicDiff

### 3. Install Dependensi MosaicDiff & Gradio

In [ ]:
!pip install -q gradio av opencv-python mmengine huggingface_hub onnxruntime-gpu

### 4. Unduh Model & Weights (DI-KOMENTARI / DISABLE SEMENTARA)

**Catatan:** Semua perintah pengunduhan model di bawah ini sengaja diberi tanda komentar (`#`).
Ketika Anda sudah siap mengunduh di Colab nanti, cukup hapus tanda `#` pada baris-baris yang diperlukan.
Total ukuran seluruh model adalah ~37 GB.

In [ ]:
# ==========================================================================
# CELL PENGUNDUHAN MODEL (COMMENTED OUT / NONAKTIF SEMENTARA)
# ==========================================================================
#
# !mkdir -p /content/MosaicDiff/models
# %cd /content/MosaicDiff
#
# --- 4A. Download detector (rfdetr.onnx) & custom LoRA (lora.safetensors) ---
# !wget -q -O MosaicDiff-windows.7z "https://github.com/dotaku22/MosaicDiff/releases/download/v1.0.1/MosaicDiff-windows.7z"
# !7z e MosaicDiff-windows.7z models/rfdetr.onnx models/lora.safetensors -omodels/ -y
# !rm MosaicDiff-windows.7z
#
# --- 4B. Download model publik HuggingFace (BasicVSR, Eros Max, CLIP, VAE) ---
# import threading
# from mosaicdiff.fetch import ensure_weights
# from mosaicdiff.settings import Settings
#
# settings = Settings.load()
# settings.paths["comfy_root"] = "/content/ComfyUI"
# settings.paths["comfy_python"] = "/usr/bin/python3"
# settings.save()
#
# print("Memulai unduhan model HuggingFace...")
# ensure_weights(settings, print, threading.Event())
# print("Semua model berhasil diunduh!")

### 5. Setup Reverse SSH Tunnel ke AWS VM
Tunnel ini akan menghubungkan port `7860` di Colab ke IP VM AWS Anda.

**Konfigurasi di AWS VM:**
1. Pastikan port 80 (atau port yang Anda gunakan) di-allow di AWS Security Group (Inbound Rule).
2. Di file `/etc/ssh/sshd_config` AWS VM, pastikan ada baris: `GatewayPorts yes` lalu jalankan `sudo systemctl restart ssh`.
3. Upload private key (`.pem`) Anda ke Google Colab (misalnya ke `/content/aws_key.pem`).

In [ ]:
AWS_PUBLIC_IP = "YOUR_AWS_IP_HERE"  # Masukkan Public IP EC2 AWS Anda
SSH_USER = "ubuntu"                 # Username default AWS EC2 (ubuntu/ec2-user)
PEM_PATH = "/content/aws_key.pem"   # Path file SSH Key Anda di Colab
REMOTE_PORT = 80                    # Port di AWS VM yang ingin diakses dari browser
LOCAL_PORT = 7860                   # Port Gradio WebUI di Colab

# Set permission key SSH
!chmod 600 {PEM_PATH}

# Jalankan Reverse SSH Tunnel di background
!ssh -o StrictHostKeyChecking=no -N -f -R 0.0.0.0:{REMOTE_PORT}:localhost:{LOCAL_PORT} {SSH_USER}@{AWS_PUBLIC_IP} -i {PEM_PATH}
print(f"SSH Tunnel aktif! Akses Web UI di: http://{AWS_PUBLIC_IP}:{REMOTE_PORT}")

### 6. Jalankan MosaicDiff Web Service (Gradio)
Jalankan web service pada port 7860. Jika tunnel aktif, Anda bisa langsung mengaksesnya lewat browser menggunakan IP AWS.

In [ ]:
%cd /content/MosaicDiff
# Jalankan web service
!python webui.py --host 0.0.0.0 --port 7860